        # 🎒 L00　補給站：統計工具箱
        **統計冒險之旅 2026**　｜　Day 1（09/21 一）🌄 統計之丘　｜　補給站　｜　🏅 50 XP

        📖 資料：勇者咖啡八月銷售


        ### 🎯 這一關你會學到
        - 匯入 pandas／NumPy／SciPy／seaborn／scikit-learn，確認版本
- 載入勇者咖啡八月銷售資料、看懂欄位
- 固定亂數種子、取得通關密語

        ### 🧭 闖關方式
        1. 先按下方「🧰 魔法工具箱」那一格左邊的 ▶（第一次執行 Colab 會花幾秒鐘連線）。
        2. 依序閱讀說明、執行範例、完成每個「🎯 任務」，再執行它下面的「檢查」格。
        3. 看到 ✅ 就往下一個任務；看到 ❌ 就依提示修改，再重新執行任務格與檢查格。
        4. 全部通過後，執行最下面的「🔑 通關密語」格，把密語貼回 [入口網頁](https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/)。

        > 💾 建議先點選「檔案 → 在雲端硬碟中儲存副本」，你的進度才會留在自己的 Google 雲端硬碟。
        > 🎲 這門課的答案常常是小數：任務會告訴你要把答案存進哪個變數，檢查時允許小小的誤差；切分、抽樣、模型請照題目用 `random_state=42`。

In [ ]:
#@title 🧰 魔法工具箱：先在右邊填「暱稱」，再按左邊的 ▶ 執行這一格 { display-mode: "form" }
暱稱 = "" #@param {type:"string"}
# ======================================================================
#  統計冒險之旅 2026 · 關卡檢查工具（看不懂沒關係，這一格不是今天的功課 😉）
# ======================================================================
import hashlib, unicodedata, io, sys, re, contextlib, traceback, builtins, math, warnings
warnings.filterwarnings("ignore")

_LEVEL = "L00"
_COURSE_NAMESPACE = "stats-quest-2026-datama"
_PREFIX = "SQ"
_TASKS = ["0-1", "0-2", "0-3"]
_XP_EACH = 16
_CHECKS = {}
_PASSED = builtins.__dict__.setdefault("_sq_" + _LEVEL, {})
_HINTS = {}

def _norm_name(s):
    return re.sub(r"\s+", "", unicodedata.normalize("NFKC", str(s))).lower()

def _squash(s):
    return re.sub(r"\s+", "", str(s))

def 出現(out, *subs):
    """輸出中是否（忽略空白）包含所有片段"""
    o = _squash(out)
    return all(_squash(x) in o for x in subs)

def 數字們(out):
    """抓出輸出裡所有的數字（float）"""
    return [float(x) for x in re.findall(r"-?\d+(?:\.\d+)?", str(out))]

# ---------------- 判分器 2.0 ----------------
class _Miss(Exception):
    pass

def 抓變數(ns, name, 型別=None):
    """從任務格執行後的變數取值；沒有就給友善訊息。"""
    if name not in ns:
        raise _Miss(f"我找不到變數 {name}，請確認你有把答案存進名字叫 {name} 的變數（大小寫要一樣）。")
    v = ns[name]
    if 型別 is not None and not isinstance(v, 型別):
        raise _Miss(f"{name} 的型別看起來不對（目前是 {type(v).__name__}）。")
    return v

def _num(v):
    try:
        import numpy as _np
        if hasattr(v, "item"): v = v.item()
    except Exception:
        pass
    return float(v)

def 約等於(v, 目標, 容差=None, 相對=0.01):
    """數值容差：|v-目標| <= 容差（預設為 目標 的 1%，且至少 1e-9）"""
    try:
        x = _num(v)
    except Exception:
        return False
    if x != x:   # NaN
        return False
    tol = 容差 if 容差 is not None else max(abs(目標) * 相對, 1e-9)
    return abs(x - 目標) <= tol

def 資料框像(obj, 列=None, 欄=None, 含欄位=None, 種類="DataFrame"):
    """檢查 DataFrame / Series：列數、欄數、必須包含的欄位；回傳 (ok, 訊息)"""
    import pandas as _pd
    if 種類 == "DataFrame" and not isinstance(obj, _pd.DataFrame):
        return False, f"這應該是一個 DataFrame（目前是 {type(obj).__name__}）。"
    if 種類 == "Series" and not isinstance(obj, _pd.Series):
        return False, f"這應該是一個 Series（目前是 {type(obj).__name__}）。"
    if 列 is not None and len(obj) != 列:
        return False, f"列數應該是 {列}，目前是 {len(obj)}。"
    if 欄 is not None and getattr(obj, "shape", (0, 0))[1] != 欄:
        return False, f"欄數應該是 {欄}，目前是 {obj.shape[1]}。"
    if 含欄位:
        cols = list(obj.columns) if hasattr(obj, "columns") else list(obj.index)
        missing = [c for c in 含欄位 if c not in cols]
        if missing:
            return False, "缺少欄位：" + "、".join(map(str, missing))
    return True, ""


class _NeedMoreInput(Exception):
    pass

_BUILTIN_NAMES = ("sum", "list", "dict", "set", "str", "int", "float", "max", "min", "len",
                  "print", "type", "range", "sorted", "abs", "round", "tuple", "map", "filter",
                  "open", "format", "all", "any", "zip", "bool", "next", "chr", "ord", "id")

_HIST = builtins.__dict__.setdefault("_sq_hist", [])
def _on_pre_run(*args):
    try:
        info = args[0]
        src = getattr(info, "raw_cell", None)
        if isinstance(src, str):
            _HIST.append(src)
    except Exception:
        pass
try:
    _ip = get_ipython()
    if not builtins.__dict__.get("_sq_hooked"):
        _ip.events.register("pre_run_cell", _on_pre_run)
        builtins.__dict__["_sq_hooked"] = True
except Exception:
    pass

def _history():
    try:
        ip = get_ipython()
        h = list(ip.user_ns.get("In") or ip.user_ns.get("_ih") or [])
    except Exception:
        h = list(globals().get("In") or [])
    return [c for c in (h + list(_HIST)) if isinstance(c, str)]

_CALL = re.compile(r"\s*(檢查|通關密語|全部檢查)\s*\(")

def _clean_cell(cell):
    return "\n".join(ln for ln in cell.splitlines() if not _CALL.match(ln))

def _is_mine(cell):
    s = cell.strip()
    if not s:
        return False
    if "#@title" in s or "任務定義(" in s or "_sq_" in s:
        return False
    if _CALL.match(s):
        return False
    return True

def _find_cells(tid):
    marker = "# 🎯 任務 " + tid
    marked = free = None
    im = ifree = -1
    for i, cell in enumerate(_history()):
        if not _is_mine(cell):
            continue
        if marker in cell:
            marked, im = cell, i
        elif "🎯 任務" not in cell:
            free, ifree = cell, i
    return marked, im, free, ifree

def _describe(src):
    body = [ln for ln in src.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    if not body:
        return "（空白）"
    first = body[0].strip()
    return ("%s%s（共 %d 行）" % (first[:52], "…" if len(first) > 52 else "", len(body)))

def _fig_info(_plt):
    out = []
    try:
        for n in _plt.get_fignums():
            f = _plt.figure(n)
            for ax in f.get_axes():
                out.append(dict(title=ax.get_title() or "", xlabel=ax.get_xlabel() or "", ylabel=ax.get_ylabel() or "",
                                n_lines=len(ax.lines), n_patches=len(ax.patches), n_collections=len(ax.collections),
                                legend=bool(ax.get_legend())))
    except Exception:
        pass
    return out

def _make_runner(src):
    def run(*inputs):
        feed = iter([str(x) for x in inputs])
        buf = io.StringIO()
        try:
            ns = dict(get_ipython().user_ns)
        except Exception:
            ns = dict(globals())
        run.shadowed = []
        for _n in _BUILTIN_NAMES:
            _b = getattr(builtins, _n, None)
            if _n in ns and _b is not None and ns[_n] is not _b:
                ns.pop(_n, None)
                run.shadowed.append(_n)
        def _fake_input(prompt=""):
            try:
                return next(feed)
            except StopIteration:
                raise _NeedMoreInput()
        ns["input"] = _fake_input
        ns["__name__"] = "__main__"
        try:
            import matplotlib
            import matplotlib.pyplot as _plt
            _plt.close("all"); _orig_show = _plt.show; _plt.show = lambda *a, **k: None
        except Exception:
            _plt = None
        run.figs = []
        try:
            with contextlib.redirect_stdout(buf):
                exec(compile(src, "<任務 " + _LEVEL + ">", "exec"), ns)
        finally:
            if _plt is not None:
                run.figs = _fig_info(_plt)
                _plt.show = _orig_show
                _plt.close("all")
        return buf.getvalue(), ns
    run.src = src
    run.figs = []
    return run

def 任務定義(tid, fn, 提示=""):
    _CHECKS[tid] = fn
    _HINTS[tid] = 提示

def _fix_shadowed():
    try:
        ns = get_ipython().user_ns
    except Exception:
        ns = globals()
    bad = []
    for n in _BUILTIN_NAMES:
        b = builtins.__dict__.get(n)
        if b is not None and n in ns and ns[n] is not b:
            del ns[n]
            bad.append(n)
    return bad

def _progress():
    done = 0
    total = 0
    for t in _TASKS:
        total += 1
        if _PASSED.get(t):
            done += 1
    bar = "■" * done + "□" * (total - done)
    return f"[{bar}] {done}/{total}"

def _run_check(tid, src):
    run = _make_runner(src)
    try:
        result = _CHECKS[tid](run)
    except _NeedMoreInput:
        return False, "你的程式呼叫 input() 的次數比題目預期的多，請檢查輸入的次數。", []
    except _Miss as e:
        return False, str(e), getattr(run, "shadowed", [])
    except Exception:
        tb = traceback.format_exc().strip().splitlines()[-1]
        return False, "程式執行時發生錯誤 → " + tb, getattr(run, "shadowed", [])
    ok, extra = (result, "") if isinstance(result, bool) else result
    return ok, extra, getattr(run, "shadowed", [])

def _pass(tid):
    first = not _PASSED.get(tid)
    _PASSED[tid] = True
    print(f"✅ 任務 {tid} 通過！{'+' + str(_XP_EACH) + ' XP ' if first else ''}{_progress()}")

def 檢查(tid):
    _shadow = _fix_shadowed()
    tid = builtins.str(tid)
    if tid not in _CHECKS:
        print(f"⚠️ 找不到任務 {tid} 的檢查設定。"); return
    marked, im, free, ifree = _find_cells(tid)
    if marked is None and free is None:
        print(f"❌ 這次執行階段裡，我找不到你寫的程式。")
        print(f"   👉 請先按「# 🎯 任務 {tid}」那一格左邊的 ▶ 執行它，再執行這一格。")
        print("   （如果剛剛重新啟動過執行階段，上面每一格都要重跑一次，包含最上面的魔法工具箱）")
        return
    order = []
    if marked is not None:
        order.append(("標記", marked))
    if free is not None and ifree > im:
        order.append(("最後執行", free))
    if not order:
        order = [("最後執行", free)]
    tried = []
    for kind, src in order:
        ok, extra, shadowed = _run_check(tid, _clean_cell(src))
        tried.append((kind, src, extra, shadowed))
        if ok:
            _pass(tid)
            if extra:
                print("   💬 " + str(extra))
            if _shadow:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(_shadow)} 拿來當變數名了，我已經幫你還原。")
                print("      建議換個名字（例如 total、items），不然後面的程式會出現很難懂的錯誤。")
            if kind == "最後執行":
                print(f"   ℹ️ 你的程式最上面少了「# 🎯 任務 {tid}」那一行，我是用你最後執行的那一格判分的。")
                print("      把那一行加回去，之後的檢查會更準確。")
            if shadowed:
                print(f"   ℹ️ 你之前把內建名稱 {'、'.join(shadowed)} 拿來當變數名了，判分時我先幫你還原。")
            if all(_PASSED.get(t) for t in _TASKS):
                print("🏆 本關所有任務都完成了！請執行最下面的「通關密語」那一格。")
            return
    kind, src, extra, shadowed = tried[0]
    print(f"❌ 任務 {tid} 還沒通過。{_progress()}")
    if extra:
        print("   💬 " + str(extra))
    if _HINTS.get(tid):
        print("   💡 提示：" + _HINTS[tid])
    _sh = _shadow + [n for n in shadowed if n not in _shadow]
    if _sh:
        print(f"   ⚠️ 你把內建名稱 {'、'.join(_sh)} 拿來當變數名了（我已還原），這會造成很難懂的錯誤，請改名後重跑那一格。")
    print("   🔎 我判分的是這一段程式：" + _describe(src))
    print(f"      如果這不是你剛剛寫的版本 → 確認第一行的「# 🎯 任務 {tid}」有保留，並重新執行那一格，再按檢查。")

def 全部檢查():
    """出錯或重新啟動執行階段後，重跑完所有任務格，再用這個一次驗收整關。"""
    _fix_shadowed()
    print(f"🔁 重新檢查 {_LEVEL} 的 {len(_TASKS)} 個任務…")
    todo = []
    for t in _TASKS:
        marked, im, free, ifree = _find_cells(t)
        if marked is None and free is None:
            todo.append(t)
            continue
        檢查(t)
    if todo:
        print("⏭️ 這次還沒執行過的任務：" + "、".join(todo))
        print("   先按那幾格左邊的 ▶ 執行，再回來執行 全部檢查()。")

def 通關密語():
    _fix_shadowed()
    missing = [t for t in _TASKS if not _PASSED.get(t)]
    if missing:
        print("🔒 還有任務未通過：" + "、".join(missing) + "　完成後再來拿密語吧！")
        return
    name = 暱稱.strip() if isinstance(暱稱, str) else ""
    if not name:
        name = input("請輸入你在入口網頁登錄的暱稱：").strip()
    if not name:
        print("⚠️ 暱稱不能是空白。"); return
    code = hashlib.sha256(f"{_COURSE_NAMESPACE}|{_LEVEL}|{_norm_name(name)}".encode("utf-8")).hexdigest()[:6].upper()
    print("=" * 46)
    print(f"🎉 恭喜 {name}！{_LEVEL} 通關！")
    print(f"🔑 通關密語：{_PREFIX}-{_LEVEL}-{code}")
    print("👉 回到入口網頁，把密語貼到這一關的「輸入通關密語」欄位。")
    print("=" * 46)

try:
    import numpy as _np_, pandas as _pd_
    _np_.random.seed(42)
except Exception:
    pass
print(f"🧰 魔法工具箱已準備好！本關有 {len(_TASKS)} 個任務：{'、'.join(_TASKS)}")
print("   做完每個任務後，執行它下方的「檢查」格；全部通過後執行最下方的「通關密語」。")

# ---------------- 各任務的檢查規則 ----------------
def _check_0_1(run):
    out, ns = run()
    for name in ["pd", "np", "stats", "sns", "sklearn"]:
        if name not in ns: return (False, f"沒有匯入 {name}。")
    v = 抓變數(ns, "版本")
    return (isinstance(v, str) and v.count(".") >= 1, "版本 應該是像 1.6.1 這樣的字串（sklearn.__version__）。")
任務定義("0-1", _check_0_1, 提示="版本 = sklearn.__version__")

def _check_0_2(run):
    out, ns = run()
    if int(抓變數(ns, "列數")) != 2563 or int(抓變數(ns, "欄數")) != 12: return (False, "列數／欄數請用 df.shape。")
    return (int(抓變數(ns, "散客筆數")) == 1006, "散客筆數 = df['會員編號'].isna().sum()。")
任務定義("0-2", _check_0_2, 提示="df.shape[0]、df.shape[1]；缺值筆數用 df['會員編號'].isna().sum()。")

def _check_0_3(run):
    out, ns = run()
    return (int(抓變數(ns, "總和")) == 31, "種子要是 42，總和才會和檢查器一樣（提示：31）。")
任務定義("0-3", _check_0_3, 提示="np.random.default_rng(42)；總和 = 骰子.sum()")

## 🌄 從六島到三峰
上一部你在六座島學會了 Python，最後在「勇者咖啡」當上資料分析師，幫老闆看懂七月的銷售紀錄。
這一部，老闆給你**八月的新資料**，而且問題升級了：不只想知道「發生了什麼」，還想知道**「這是真的還是巧合？」「下個月會怎樣？」**——這就是統計與機器學習要回答的問題。

三天、三座山：
| 山峰 | 你會學到 |
|---|---|
| 🌄 統計之丘（Day 1） | 看懂資料：敘述統計、分布、抽樣與檢定 |
| ⛰️ 模型之嶺（Day 2） | 建立模型：迴歸、過度擬合、分類與評估 |
| 🗻 預測之巔（Day 3） | 讓模型可靠：交叉驗證、集成、正規化、分群；最終專題 |

## 🎒 0-1　統計工具箱
這門課的裝備都已經裝在 Colab 裡，不用安裝，只要 `import`：

| 套件 | 綽號 | 一句話 |
|---|---|---|
| `pandas`（pd） | 會寫程式的 Excel | 讀資料、篩選、分組、摘要（第一部學過） |
| `numpy`（np） | 整籠一起算 | 陣列運算、亂數 |
| `scipy.stats`（stats） | 統計公式庫 | 常態分布、t 檢定、卡方檢定……不用自己推公式 |
| `seaborn`（sns） | 漂亮的圖 | 一行畫出直方圖、盒鬚圖 |
| `scikit-learn`（sklearn） | 模型工廠 | Day 2 起用它做迴歸、分類、交叉驗證 |

In [ ]:
import pandas as pd, numpy as np
from scipy import stats
import seaborn as sns, matplotlib.pyplot as plt
import sklearn
print("pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", sklearn.__version__)

## 0-2　勇者咖啡八月資料
八月三家分店（信義店、板橋店、中壢店）的每一筆訂單，比七月多了三個欄位：**時段、天氣、會員編號**（沒有會員編號的是散客）。
網址：`https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.1/data/coffee_sales_aug.csv`

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.1/data/coffee_sales_aug.csv")
print(df.shape)          # (列數, 欄數)
df.head()

In [ ]:
df.info()                # 每欄的型別與非缺值筆數：會員編號有缺值 = 散客

## 0-3　亂數種子：讓「隨機」可以重現
統計課常常要「隨機抽樣」「隨機模擬」。電腦的亂數其實是照一串公式算出來的，只要**種子（seed）**一樣，跑出來就一模一樣——像同一份樂透號碼的產生器。
這門課所有抽樣、切分、模型都用 `42` 當種子，你的答案才會和檢查器一致。

In [ ]:
rng = np.random.default_rng(42)        # 建立一個「種子 42」的亂數產生器
print(rng.integers(1, 7, size=10))      # 擲 10 次骰子（1～6）
rng = np.random.default_rng(42)
print(rng.integers(1, 7, size=10))      # 再建一次同樣的種子 → 一模一樣

### 🎯 任務 0-1　匯入工具箱

匯入 `pandas`（取名 `pd`）、`numpy`（`np`）、`scipy.stats`（`stats`）、`seaborn`（`sns`）、`sklearn`，並把 scikit-learn 的版本字串存成 `版本`。

In [ ]:
# 🎯 任務 0-1　匯入工具箱（請保留這一行）
import pandas as pd, numpy as np
from scipy import stats
import seaborn as sns
import sklearn
版本 = ???
print("scikit-learn 版本：", 版本)

In [ ]:
檢查("0-1")   # ◀ 執行這一格，看看任務 0-1 有沒有過關

### 🎯 任務 0-2　載入八月資料

讀取八月銷售 CSV 成 `df`，把列數存成 `列數`、欄數存成 `欄數`，並把**散客筆數**（會員編號是缺值的筆數）存成 `散客筆數`。

In [ ]:
# 🎯 任務 0-2　載入八月資料（請保留這一行）
df = pd.read_csv("https://raw.githubusercontent.com/johnnychao/stats-quest-2026/v1.0.0-rc.1/data/coffee_sales_aug.csv")
列數 = ???
欄數 = ???
散客筆數 = ???
print(列數, 欄數, 散客筆數)

In [ ]:
檢查("0-2")   # ◀ 執行這一格，看看任務 0-2 有沒有過關

### 🎯 任務 0-3　固定亂數種子

用種子 42 建立亂數產生器 `rng`，擲 10 次骰子存成 `骰子`（`rng.integers(1, 7, size=10)`），把總和存成 `總和`。

In [ ]:
# 🎯 任務 0-3　固定亂數種子（請保留這一行）
rng = np.random.default_rng(???)
骰子 = rng.integers(1, 7, size=10)
總和 = ???
print(骰子, 總和)

In [ ]:
檢查("0-3")   # ◀ 執行這一格，看看任務 0-3 有沒有過關

---
## 🔑 通關密語
　補給站完成，你已經站在山腳下了。
全部任務都 ✅ 之後，執行下面這一格，會得到你專屬的通關密語（和暱稱綁定，每個人不一樣）。

In [ ]:
通關密語()

---
### 🧭 接下來
**下一關：📊 L01 看懂資料的長相** → [在 Colab 開啟](https://colab.research.google.com/github/johnnychao/stats-quest-2026/blob/v1.0.0-rc.1/notebooks/L01_describe.ipynb)

回到入口網頁：https://johnnychao.github.io/stats-quest-2026/rc/v1.0.0-rc.1/